In [1]:
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

In [2]:
df_manifest = pd.read_csv("../data/manifest_crop.csv")
sample = df_manifest.iloc[0]

img_path = f"../data/preprocessed/{sample['label']}/{sample['filename']}"
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# GLCM dihitung di 4 arah (0, 45, 90, 135 derajat) lalu dirata-rata,
# karena orientasi biji jagung di foto tidak selalu sama
glcm = graycomatrix(
    gray,
    distances=[1],
    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
    levels=256,
    symmetric=True,
    normed=True
)

contrast    = graycoprops(glcm, 'contrast').mean()
correlation = graycoprops(glcm, 'correlation').mean()
energy      = graycoprops(glcm, 'energy').mean()
homogeneity = graycoprops(glcm, 'homogeneity').mean()

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h_mean = hsv[:, :, 0].mean()
s_mean = hsv[:, :, 1].mean()
v_mean = hsv[:, :, 2].mean()

print(f"File     : {sample['filename']}")
print(f"Label    : {sample['label']}")
print(f"Contrast : {contrast:.4f}")
print(f"Correlation : {correlation:.4f}")
print(f"Energy   : {energy:.4f}")
print(f"Homogeneity : {homogeneity:.4f}")
print(f"Hue mean : {h_mean:.2f}")
print(f"Saturation mean : {s_mean:.2f}")
print(f"Value mean : {v_mean:.2f}")

File     : normal_001_biji1.jpg
Label    : sehat
Contrast : 25.8421
Correlation : 0.9964
Energy   : 0.0321
Homogeneity : 0.3610
Hue mean : 30.68
Saturation mean : 106.61
Value mean : 149.34


In [3]:
fitur_list = []

for idx, row in df_manifest.iterrows():
    img_path = f"../data/preprocessed/{row['label']}/{row['filename']}"
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    glcm = graycomatrix(
        gray,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )
    contrast    = graycoprops(glcm, 'contrast').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    energy      = graycoprops(glcm, 'energy').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h_mean = hsv[:, :, 0].mean()
    s_mean = hsv[:, :, 1].mean()
    v_mean = hsv[:, :, 2].mean()

    fitur_list.append({
        "filename": row['filename'],
        "label": row['label'],
        "contrast": contrast,
        "correlation": correlation,
        "energy": energy,
        "homogeneity": homogeneity,
        "hue": h_mean,
        "saturation": s_mean,
        "value": v_mean
    })

df_fitur = pd.DataFrame(fitur_list)
df_fitur.to_csv("../data/manifest_fitur.csv", index=False)

print(f"Total gambar diproses: {len(df_fitur)}")
df_fitur.head()

Total gambar diproses: 207


,filename,label,contrast,correlation,energy,homogeneity,hue,saturation,value
0,normal_001_biji1.jpg,sehat,25.842051,0.996368,0.032105,0.361015,30.675049,106.611755,149.340332
1,normal_002_biji1.jpg,sehat,22.805198,0.997290,0.027281,0.327272,32.918701,63.728821,148.067017
2,normal_002_biji2.jpg,sehat,20.040370,0.996972,0.029912,0.343545,30.887756,94.991821,151.778564
3,normal_002_biji3.jpg,sehat,23.656310,0.996817,0.026988,0.317613,38.722046,77.195618,137.762207
4,normal_004_biji1.jpg,sehat,20.507643,0.997156,0.030847,0.348835,32.742859,83.113403,150.955139
